# 06. Ponte verso la fase agente e RL

La seconda fase del progetto usa i decision model dentro un agente. Un decision model e veloce ed economico, quindi funziona bene come router o guard: sceglie la prossima azione, mentre codice deterministico applica le regole rigide e le decisioni incerte vengono passate a un modello piu forte.

Inquadriamo un piccolo task sequenziale come problema in stile RL. La ricompensa e il successo del task meno un costo per ogni passo. Il mock ci permette di simulare una politica appresa senza download o chiavi.


<div align='center'><img src='../images/agent_loop.png' alt='Loop agente' style='width:80%; max-width:850px;'></div>

*Il decision model valuta le azioni, il codice deterministico applica regole e soglie, e l'ambiente restituisce un nuovo stato più una ricompensa.*


## Passo 1. Definire l'insieme delle azioni

Il nostro agente didattico ha tre azioni. Dichiararle esplicitamente e la stessa idea di una domanda Choice: il modello puo scegliere solo cio che elenchiamo.


In [ ]:
ACTIONS = ['retrieve', 'answer', 'escalate']
print(ACTIONS)
assert len(ACTIONS) == 3


## Passo 2. Una politica appresa simulata

Facciamo preferire al mock `retrieve` quando non ci sono evidenze e `answer` quando ci sono. Questo sostituisce un modello addestrato su tracce reali. L'indice gold e qui solo un espediente didattico.


In [ ]:
from decision_qa import MockClient, ChoiceQuestion
client = MockClient(seed=7)
state = {'question': 'What did the trial report?', 'has_evidence': False}
print(state)


In [ ]:
def policy_question(state):
    target = 0 if not state['has_evidence'] else 1
    return ChoiceQuestion(label='next_action', options=ACTIONS, gold_index=target)


In [ ]:
q0 = policy_question(state)
print(q0.options[q0.gold_index])


In [ ]:
res0 = client.decision(state, [q0])[0]
print(res0.probabilities)


In [ ]:
action0 = max(res0.probabilities, key=res0.probabilities.get)
print(action0)
assert action0 == 'retrieve'


## Passo 3. Le regole deterministiche vengono prima

Un decision model non va fidato per i vincoli rigidi. Aggiungiamo una regola che vieta di rispondere senza evidenze. Il codice delle regole e economico, testabile e sempre corretto.


In [ ]:
def enforce_rules(state, action):
    if action == 'answer' and not state['has_evidence']:
        return 'retrieve'
    return action


In [ ]:
print(enforce_rules(state, 'answer'))
assert enforce_rules(state, 'answer') == 'retrieve'


In [ ]:
state['has_evidence'] = True
print(enforce_rules(state, 'answer'))
assert enforce_rules(state, 'answer') == 'answer'


## Passo 4. La soglia di confidenza

Se la probabilita massima e sotto una soglia, passiamo oltre invece di agire. Creiamo un mock senza informazioni gold per produrre una confidenza bassa e uniforme.


In [ ]:
uniform_client = MockClient(seed=1, quality=0.0)
uncertain = ChoiceQuestion(label='next_action', options=ACTIONS)
res_u = uniform_client.decision(state, [uncertain])[0]
print(res_u.probabilities)


In [ ]:
def gate(result, threshold=0.6):
    action = max(result.probabilities, key=result.probabilities.get)
    confidence = result.probabilities[action]
    return 'escalate' if confidence < threshold else action


In [ ]:
print(gate(res_u))
assert gate(res_u) == 'escalate'


## Passo 5. Un passo dell'agente

Un passo combina le tre parti: chiedere al modello, applicare le regole, applicare la soglia. Lo scriviamo come una piccola funzione e lo testiamo su uno stato senza evidenze.


In [ ]:
def agent_step(state):
    q = policy_question(state)
    result = client.decision(state, [q])[0]
    action = max(result.probabilities, key=result.probabilities.get)
    action = enforce_rules(state, action)
    action = gate(result) if result.probabilities[action] < 0.6 else action
    return action, result.probabilities[action]


In [ ]:
fresh = {'question': 'What did the trial report?', 'has_evidence': False}
action, conf = agent_step(fresh)
print(action, round(conf, 3))
assert action == 'retrieve'


## Passo 6. Eseguire un episodio completo

Un episodio ripete i passi finche l'agente risponde, passa oltre o esaurisce i passi. Il recupero imposta il flag delle evidenze. La risposta restituisce una ricompensa ridotta dal numero di passi.


In [ ]:
def episode(max_steps=4):
    s = {'question': 'What did the trial report?', 'has_evidence': False}
    for step in range(1, max_steps + 1):
        action, _ = agent_step(s)
        if action == 'retrieve':
            s['has_evidence'] = True
        elif action == 'answer':
            return 1.0 - 0.1 * step
        else:
            return -0.2
    return -0.1


In [ ]:
reward = episode()
print(round(reward, 3))
assert -1.0 <= reward <= 1.0


In [ ]:
rewards = [episode() for _ in range(20)]
mean_reward = sum(rewards) / len(rewards)
print(round(mean_reward, 3))


Il grafico della ricompensa per episodio mostra se la politica e stabile. Una linea piatta a 0.8 significa che l'agente recupera una volta e poi risponde, mentre i picchi negativi indicano escalation. La linea tratteggiata e la ricompensa media.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.2, 3.2), constrained_layout=True)
ax.plot(range(1, len(rewards) + 1), rewards, 'o-', color='#54A24B', label='episode reward')
ax.axhline(mean_reward, color='#8A5A00', linestyle='--', label=f'mean {mean_reward:.2f}')
ax.set_xlabel('Episode', labelpad=6)
ax.set_ylabel('Reward', labelpad=8)
ax.set_title('Reward per episode')
ax.set_ylim(-0.4, 1.0)
ax.grid(alpha=0.25)
ax.legend(loc='lower right', fontsize=8)
plt.show()


## Passo 7. Il vocabolario RL

Lo stato e cio che l'agente osserva. L'azione e il risultato tipizzato della Choice. La politica e il decision model piu regole e soglia. La ricompensa valuta l'esito. In un vero task della Fase 2 sostituiremmo il mock con un modello open locale e definiremmo una ricompensa reale, per esempio exact match meno costo di latenza su un benchmark QA multi-hop.

Progetti gratuiti da studiare: JevLoop e ProgressGate, entrambi open source e documentati nella lista curata awesome-decision-models. Ogni strumento usato in questo repository e gratuito.
